# Qima HR Payroll Ingestion Pipeline -- SharePoint to Snowflake RAW Layer

## What this notebook does

This pipeline moves payroll Excel files from Qima's SharePoint folder into Snowflake, extracting their raw cell content into a structured JSON format. It is the first stage of the payroll data platform -- everything downstream (transformation, modelling, reporting) consumes from the `FILE_LOAD` table this pipeline writes to.

## Workflow

| Step | Cell | What it does | Why |
|---|---|---|---|
| 1 | Context | Sets the warehouse, database, and schema | Ensures the notebook targets the correct Snowflake objects regardless of session defaults |
| 2 | SharePoint config + auth | Defines connection parameters and obtains a bearer token via OAuth2 client credentials | All SharePoint API calls in later steps depend on this token. Config is inline so no external procedure is needed |
| 3 | Create temporary stage | Creates a session-scoped stage (`TEMP_PAYROLL_STAGE`) | Raw file bytes land here temporarily. Session-scoped means it auto-drops when the notebook finishes -- no sensitive payroll files persist in Snowflake beyond the run |
| 4 | list_children() | Fetches items at a single Graph API folder URL, handling pagination | SharePoint can return results across multiple pages. This is the low-level API primitive |
| 5 | walk_folder() | Recursively traverses all subfolders under the payroll root | Each subsidiary has its own subfolder. This flattens the entire tree into one list of files |
| 6 | list_candidates() | Computes the ingestion window and filters to new/modified .xlsx files | Only picks up files modified after the last successful ingest and before the safety buffer. Prevents re-ingesting already-processed files |
| 7 | download_files() | Downloads each candidate from SharePoint via the Graph API | Fetches the actual file bytes. Each file is handled independently -- one failure does not abort the batch |
| 8 | stage_files() | PUTs downloaded bytes into the temporary Snowflake stage | Bytes go straight from memory to the stage. They never touch local disk |
| 9 | INSERT into FILE_LOAD | Logs every candidate into FILE_LOAD with its outcome | Every file gets a durable row -- SUCCESS or FAILED. Nothing is silently dropped. Previous versions of the same file are flagged `IS_CURRENT = FALSE` |
| 10 | Extract and update | Parses each staged .xlsx into a nested JSON grid and stores it in RAW_CONTENT | Raw cells only -- no header detection, no layout interpretation. All structural logic lives in the transformation layer |

## Key rules

- **Ingestion is deliberately dumb.** It never looks inside a file. It only moves bytes. This means it cannot break when Qima changes their Excel template.
- **Extraction is deliberately dumb.** It only converts the grid to JSON, cell by cell, sheet by sheet. No header detection, no layout interpretation. That is transformation's job.
- **Nothing is silently dropped.** Every candidate file gets a FILE_LOAD row, whether it succeeded or failed. Failed files show up in monitoring and stay eligible for retry.
- **Raw files are never persisted.** The temporary stage auto-drops at session end. Payroll data in file form only exists for the seconds it takes to parse it.
- **Versioning via IS_CURRENT.** When HR corrects and re-uploads a file, the previous version stays in FILE_LOAD (flagged `IS_CURRENT = FALSE`) and the new version is inserted as `IS_CURRENT = TRUE`. Full audit trail, no data loss.
- **Idempotent on re-run.** The ingestion window uses exclusive bounds `(FROM_TS, TO_TS)` so already-processed files are never picked up again. The extraction cell skips rows already marked SUCCESS.

## HR scenarios handled

1. **New upload** -- file is ingested, extracted, and stored with `IS_CURRENT = TRUE`.
2. **Correction / re-upload** -- previous version flagged `IS_CURRENT = FALSE`, new version inserted as `IS_CURRENT = TRUE`. Both preserved for audit.
3. **Deletion** -- pipeline doesn't see the file, takes no action. Existing rows preserved.
4. **No activity** -- pipeline runs, finds zero candidates, finishes cleanly as a no-op.

## 1. Context

In [ ]:
!pip install openpyxl

In [ ]:
%%sql -r dataframe_1
USE WAREHOUSE SANDBOX_WH;
USE DATABASE SANDBOX_DB;
USE SCHEMA HR_PAYROLL_QIMA;

## 2. Get an Access Token

In [ ]:
# SharePoint config + auth

import requests
import io
from urllib.parse import quote

GRAPH       = 'https://graph.microsoft.com/v1.0'
TENANT_ID   = '77fc8d6c-15ec-4aea-9bd6-cf77b407a763'
CLIENT_ID   = 'f1343367-3e3a-4ab0-8438-e72f261a6994'
TOKEN_URL   = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'

SITE_ID     = 'learnfabricsbi.sharepoint.com,5cfbd818-9daa-43e7-9676-7e69ccdbd7a0,859f7104-aaf6-4d0b-8cd3-30a1bc521983'
DRIVE_ID    = 'b!GNj7XKqd50OWdn5pzNvXoARxn4X2qgtNjNMwobxSGYOsH0CrHyfVRYKr4I5Z_tXk'
FOLDER_PATH = 'Payroll files'
STAGE       = 'TEMP_PAYROLL_STAGE'


SECRET_PATH = '/secrets/sandbox_db/hr_payroll_qima/sharepoint_hr_payroll_client_secret/secret_string'

def get_token() -> str:
    with open(SECRET_PATH) as f:
        secret = f.read().strip()
    resp = requests.post(TOKEN_URL, data={
        'client_id':     CLIENT_ID,
        'client_secret': secret,
        'scope':         'https://graph.microsoft.com/.default',
        'grant_type':    'client_credentials',
    }, timeout=30)
    if resp.status_code != 200:
        # Status only - the body can echo request parameters back.
        raise RuntimeError(f'Token request failed: HTTP {resp.status_code}')
    return resp.json()['access_token']


# ponytail: one token for the whole run; re-run this cell if a run outlives the 1h expiry.
HDR = {'Authorization': f'Bearer {get_token()}'}
print('token: OK - Valid for an hour')

## 3. Create Temporary Stage

In [ ]:
%%sql -r dataframe_2
CREATE TEMPORARY STAGE IF NOT EXISTS TEMP_PAYROLL_STAGE
    COMMENT = 'Session-scoped stage for raw payroll file bytes. Auto-dropped at session end.';

## 4. List SharePoint Files

In [ ]:
def list_children(url):
    """Fetch all children at a Graph API URL, following pagination."""
    items = []
    while url:
        r = requests.get(url, headers=HDR, timeout=60)
        if r.status_code != 200:
            raise RuntimeError(f'Graph list failed: HTTP {r.status_code} - {r.text[:300]}')
        data = r.json()
        items.extend(data.get('value', []))
        url = data.get('@odata.nextLink')
    return items


# Verify: list the top-level items inside the payroll folder.
folder = quote(FOLDER_PATH, safe='/')
root_url = f'{GRAPH}/sites/{SITE_ID}/drives/{DRIVE_ID}/root:/{folder}:/children'
top_items = list_children(root_url)

print(f'{len(top_items)} item(s) in /{FOLDER_PATH}:\n')
for item in top_items:
    kind = 'folder' if 'folder' in item else 'file'
    print(f"  [{kind}] {item['name']}")

In [ ]:
def walk_folder(folder_id, depth=0, max_depth=5):
    """Recursively collect every file under a folder. Subsidiaries file into subfolders."""
    if depth > max_depth:
        return []
    files = []
    for item in list_children(f'{GRAPH}/drives/{DRIVE_ID}/items/{folder_id}/children'):
        if 'folder' in item:
            files.extend(walk_folder(item['id'], depth + 1, max_depth))
        else:
            files.append(item)
    return files


# Verify: walk the entire payroll folder tree and show all files found.
folder = quote(FOLDER_PATH, safe='/')
r = requests.get(f'{GRAPH}/sites/{SITE_ID}/drives/{DRIVE_ID}/root:/{folder}',
                 headers=HDR, timeout=60)
if r.status_code != 200:
    raise RuntimeError(f'Folder lookup failed: HTTP {r.status_code} - {r.text[:300]}')

all_files = walk_folder(r.json()['id'])

# # List each raw details of each file from SharePoint
# print(f'{len(all_files)} file(s) found across all subfolders:\n')
# for f in all_files:
#     print(json.dumps(f, indent=2))
#     print()

def fmt_size(n):
    """Bytes -> something a human reads at a glance."""
    return f'{n / 1048576:.1f} MB' if n >= 1048576 else f'{n / 1024:.0f} KB'


def fmt_ts(s):
    return datetime.fromisoformat(s.replace('Z', '+00:00')).strftime('%d %b %Y, %I:%M %p UTC')


print(f'{len(all_files)} file(s) found under /{FOLDER_PATH}\n')
for i, f in enumerate(all_files, 1):
    folder = f.get('parentReference', {}).get('name', '?')
    if folder == FOLDER_PATH:
        folder += '   <-- root folder, not a subsidiary submission'
    saved_by = f.get('lastModifiedBy', {}).get('user', {}).get('displayName', 'unknown')
    added_by = f.get('createdBy', {}).get('user', {}).get('displayName', 'unknown')
    print(f"[{i}] {f['name']}")
    print(f"    FOLDER_NAME : {folder}")
    print(f"    Size           : {fmt_size(f.get('size', 0))}")
    print(f"    SHAREPOINT_MODIFED_AT  : {fmt_ts(f['lastModifiedDateTime'])} ")
    print(f"    SHAREPOINT_MODIFIED_BY : {saved_by}")
    print(f"    SHAREPOINT_CREATED_AT : {fmt_ts(f['createdDateTime'])}  ")
    print(f"    SHAREPOINT_CREATED_BY : {added_by}")
    print(f"    Content hash   : {f.get('file', {}).get('hashes', {}).get('quickXorHash', '-')}")
    print(f"    SharePoint ID  : {f['id']}")
    print()



In [ ]:
from datetime import datetime, timedelta, timezone
from snowflake.snowpark.context import get_active_session

session = get_active_session()

def list_candidates(session, all_files, offset_minutes=1):

    row = session.sql("""
        SELECT COALESCE(
            MAX(INGESTED_AT),
            '2024-01-01'::TIMESTAMP_TZ
        ) AS HWM
        FROM FILE_LOAD
        WHERE INGEST_STATUS = 'SUCCESS'
    """).collect()
    
    from_ts = row[0]['HWM']
    to_ts = session.sql(f"""
        SELECT TIMESTAMPADD('MINUTE', -{offset_minutes}, CONVERT_TIMEZONE('UTC', CURRENT_TIMESTAMP()))
    """).collect()[0][0]
    
    matched = []
    for item in all_files:
        # List only the items with .xlsx file format
        if not item.get('name', '').endswith('.xlsx'): 
            continue
        mod = datetime.fromisoformat(item['lastModifiedDateTime'].replace('Z', '+00:00'))
        if from_ts < mod < to_ts:
            matched.append({
                'id':          item['id'],
                'name':        item['name'],
                'path':        item.get('parentReference', {}).get('path', ''),
                'modified_at': item['lastModifiedDateTime'],
                'modified_by': item.get('lastModifiedBy', {}).get('user', {}).get('displayName'),
                'created_at':  item.get('createdDateTime'),
                'created_by':  item.get('createdBy', {}).get('user', {}).get('displayName'),
                'size_bytes':  item.get('size'),
            })
    return matched, from_ts, to_ts


candidates, FROM_TS, TO_TS = list_candidates(session, all_files)

print(f'{len(candidates)} files in window: [LAST_INGESTED_AT: {FROM_TS}, CURRENT_TIMESTAMP: {TO_TS})\n')
for i, c in enumerate(candidates, 1):
    print(f"  [{i}] {c['name']}")
    print(f"      id          : {c['id']}")
    print(f"      path        : {c.get('path', 'N/A')}")
    print(f"      size_bytes  : {c['size_bytes']:,}")
    print(f"      modified_at : {fmt_ts(c['modified_at'])}  by {c.get('modified_by', 'N/A')}")
    print(f"      modified_by : {c.get('modified_by', 'N/A')}")
    print(f"      created_at  : {fmt_ts(c['created_at'])}  by {c.get('created_by', 'N/A')}")
    print(f"      created_by : {c.get('created_by', 'N/A')}")
    print()

if not candidates:
    print('No new/modified files to ingest - pipeline will finish cleanly.')

## 5. Download the SharePoint Files

In [ ]:
def download_files(candidates):
    """Download each candidate file from SharePoint via Graph API. Returns a list of dicts with name, bytes or error."""
    downloads = []
    for c in candidates:
        try:
            r = requests.get(f"{GRAPH}/drives/{DRIVE_ID}/items/{c['id']}/content",
                             headers=HDR, timeout=120)
            if r.status_code != 200:
                downloads.append({'name': c['name'], 'status': 'FAILED', 'error': f'HTTP {r.status_code}'})
                continue
            downloads.append({'name': c['name'], 'status': 'OK', 'content': r.content})
        except Exception as e:
            downloads.append({'name': c['name'], 'status': 'FAILED', 'error': f'{type(e).__name__}: {e}'})
    return downloads


downloads = download_files(candidates)

ok = [d for d in downloads if d['status'] == 'OK']
failed = [d for d in downloads if d['status'] == 'FAILED']

print(f'Downloaded {len(ok)}/{len(candidates)} file(s)\n')
for d in ok:
    print(f"  [OK]   {d['name']}  ({len(d['content']):,} bytes)")
for d in failed:
    print(f"  [FAIL] {d['name']}  ({d['error']})")

## 

## 6. PUT files into STAGE

In [ ]:
def stage_files(downloads, stage_name):
    """PUT each successfully downloaded file into the Snowflake temporary stage."""
    results = []
    for d in downloads:
        if d['status'] != 'OK':
            results.append({'name': d['name'], 'status': 'SKIPPED', 'reason': d.get('error', 'download failed')})
            continue
        try:
            session.file.put_stream(io.BytesIO(d['content']), f"@{stage_name}/{d['name']}",
                                    auto_compress=False, overwrite=True)
            results.append({'name': d['name'], 'status': 'OK', 'bytes': len(d['content'])})
        except Exception as e:
            results.append({'name': d['name'], 'status': 'FAILED', 'reason': f'{type(e).__name__}: {e}'})
    return results


stage_results = stage_files(downloads, STAGE)

staged = [r for r in stage_results if r['status'] == 'OK']
print(f'Staged {len(staged)}/{len(downloads)} file(s)\n')
for r in stage_results:
    if r['status'] == 'OK':
        print(f"  [OK]      {r['name']}  ({r['bytes']:,} bytes)")
    elif r['status'] == 'SKIPPED':
        print(f"  [SKIPPED] {r['name']}  ({r['reason']})")
    else:
        print(f"  [FAIL]    {r['name']}  ({r['reason']})")

# Verify what's on stage
print('\n--- Staged files ---')
for sf in session.sql(f'LIST @{STAGE}').collect():
    print(f"  {sf['name']}  ({sf['size']:,} bytes)")

## 7. INSERT into a FILE_LOAD table

In [ ]:
import uuid

RUN_ID = str(uuid.uuid4())

# Build a lookup from stage_results keyed by file name.
stage_by_name = {r['name']: r for r in stage_results}

rows_inserted = 0

for c in candidates:
    sr = stage_by_name.get(c['name'], {})
    staged_ok = sr.get('status') == 'OK'

    ingest_status = 'SUCCESS' if staged_ok else 'FAILED'
    error_message = None if staged_ok else sr.get('reason', 'unknown error')

    # Mark previous versions of this file as not current.
    session.sql("""
        UPDATE FILE_LOAD SET IS_CURRENT = FALSE
        WHERE SHAREPOINT_ITEM_ID = :1 AND IS_CURRENT = TRUE
    """, params=[c['id']]).collect()

    session.sql("""
        INSERT INTO FILE_LOAD (
            RUN_ID, FILE_NAME, FILE_PATH, SHAREPOINT_ITEM_ID, SHAREPOINT_MODIFIED_AT,
            SHAREPOINT_MODIFIED_BY, SHAREPOINT_CREATED_AT, SHAREPOINT_CREATED_BY,
            FILE_SIZE_BYTES, INGESTED_AT, INGEST_STATUS, ERROR_MESSAGE, IS_CURRENT
        )
        SELECT
            :1, :2, :3, :4,
            :5, :6, :7,
            :8, :9, CURRENT_TIMESTAMP(), :10, :11, TRUE
    """, params=[
        RUN_ID,
        c['name'],
        c.get('path'),
        c['id'],
        c['modified_at'],
        c.get('modified_by'),
        c.get('created_at'),
        c.get('created_by'),
        c.get('size_bytes'),
        ingest_status,
        error_message,
    ]).collect()

    rows_inserted += 1

print(f'Inserted {rows_inserted} FILE_LOAD row(s) for RUN_ID = {RUN_ID}\n')

# Verify
for row in session.sql(f"""
    SELECT FILE_NAME, INGEST_STATUS, IS_CURRENT, ERROR_MESSAGE
    FROM FILE_LOAD WHERE RUN_ID = '{RUN_ID}'
""").collect():
    icon = 'OK' if row['INGEST_STATUS'] == 'SUCCESS' else 'FAIL'
    msg = f"  ({row['ERROR_MESSAGE']})" if row['ERROR_MESSAGE'] else ''
    print(f"  [{icon}] {row['FILE_NAME']}  IS_CURRENT={row['IS_CURRENT']}{msg}")

## 8. Extract and Update

In [ ]:
import json
from openpyxl import load_workbook


def parse_workbook_to_json(file_bytes):
    """Open an xlsx from bytes, walk every sheet, return a nested JSON grid.
    No header detection, no layout interpretation — just raw cell values.
    Structure: { "SheetName": [[row1_cells...], [row2_cells...], ...], ... }
    """
    wb = load_workbook(io.BytesIO(file_bytes), read_only=True, data_only=True)
    workbook_data = {}
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows():
            cells = []
            for cell in row:
                val = cell.value
                if val is None:
                    cells.append(None)
                elif isinstance(val, datetime):
                    cells.append(val.isoformat())
                elif isinstance(val, (int, float, bool)):
                    cells.append(val)
                else:
                    cells.append(str(val))
            rows.append(cells)
        workbook_data[sheet_name] = rows
    wb.close()
    return workbook_data


# Only process rows from this run that were successfully staged but not yet extracted.
success_rows = session.sql(f"""
    SELECT LOAD_ID, FILE_NAME
    FROM FILE_LOAD
    WHERE RUN_ID = '{RUN_ID}'
      AND INGEST_STATUS = 'SUCCESS'
      AND EXTRACT_STATUS = 'NOT_ATTEMPTED'
""").collect()

if not success_rows:
    print('Nothing to extract — all rows already processed or no successful ingests.')


extracted = 0
failed = 0

for row in success_rows:
    load_id = row['LOAD_ID']
    file_name = row['FILE_NAME']
    try:
        staged_stream = session.file.get_stream(f"@{STAGE}/{file_name}")
        file_bytes = staged_stream.read()
        raw_content = parse_workbook_to_json(file_bytes)

        session.sql("""
            UPDATE FILE_LOAD
            SET EXTRACT_STATUS = 'SUCCESS',
                EXTRACTED_AT   = CURRENT_TIMESTAMP(),
                RAW_CONTENT    = PARSE_JSON(:1)
            WHERE LOAD_ID = :2
        """, params=[json.dumps(raw_content), load_id]).collect()

        sheet_summary = ', '.join(f"{k} ({len(v)} rows)" for k, v in raw_content.items())
        print(f"  [OK]   {file_name}  ->  {sheet_summary}")
        extracted += 1

    except Exception as e:
        session.sql("""
            UPDATE FILE_LOAD
            SET EXTRACT_STATUS = 'FAILED',
                EXTRACTED_AT   = CURRENT_TIMESTAMP(),
                ERROR_MESSAGE  = :1
            WHERE LOAD_ID = :2
        """, params=[f'{type(e).__name__}: {e}', load_id]).collect()

        print(f"  [FAIL] {file_name}  ->  {type(e).__name__}: {e}")
        failed += 1

print(f"\nExtraction complete: {extracted} succeeded, {failed} failed, "
      f"{len(candidates) - len(success_rows)} not attempted (ingest failed)")

In [ ]:
# Print RAW_CONTENT for a specific file — change the LOAD_ID to inspect a different file.
INSPECT_LOAD_ID = 205

row = session.sql(f"""
    SELECT FILE_NAME, RAW_CONTENT
    FROM FILE_LOAD
    WHERE LOAD_ID = {INSPECT_LOAD_ID}
""").collect()

if not row:
    print(f'No row found for LOAD_ID = {INSPECT_LOAD_ID}')
else:
    print(f"File: {row[0]['FILE_NAME']}\n")
    import json
    raw = json.loads(row[0]['RAW_CONTENT'])
    print(json.dumps(raw, indent=2, ensure_ascii=False))